In [1]:
# import lib 
import os
import joblib
import numpy as np
import pandas as pd

In [2]:
# Define artifact paths

model_path = "../artifacts/model_trainer/model.pkl"
scaler_path = "../artifacts/model_trainer/scaler.pkl"

In [3]:
# Check the files

print("Model exists:", os.path.exists(model_path))
print("Scaler exists:", os.path.exists(scaler_path))

Model exists: True
Scaler exists: True


In [4]:
# load the files

model = joblib.load(model_path)
scaler = joblib.load(scaler_path)

print("Model:", model)
print("Scaler:", scaler)

Model: GaussianMixture(covariance_type='spherical', n_components=2, n_init=20,
                random_state=42)
Scaler: StandardScaler()


In [5]:
# Create new customer RFM data

new_customer = pd.DataFrame({
    "Recency": [25],
    "Frequency": [8],
    "Monetary": [4200]
})

new_customer

,Recency,Frequency,Monetary
0,25,8,4200


In [6]:
# Apply log transformation

new_customer_log = np.log1p(new_customer)

new_customer_log

,Recency,Frequency,Monetary
0,3.258097,2.197225,8.343078


In [7]:
# Apply the saved scaler

new_customer_scaled = scaler.transform(new_customer_log)

new_customer_scaled

array([[-0.42730728,  1.24686844,  1.39436188]])

In [8]:
# Convert scaled data to DataFrame

new_customer_scaled = pd.DataFrame(
    new_customer_scaled,
    columns=new_customer.columns
)

new_customer_scaled

,Recency,Frequency,Monetary
0,-0.427307,1.246868,1.394362


In [9]:
# Predict the customer cluster

cluster = model.predict(new_customer_scaled)

print("Predicted Cluster:", cluster[0])

Predicted Cluster: 1


In [10]:
# Get membership probabilities

probabilities = model.predict_proba(new_customer_scaled)

probabilities

array([[2.36083164e-04, 9.99763917e-01]])

In [11]:
# Get probability of the predicted cluster

predicted_cluster = cluster[0]
membership_probability = probabilities[0, predicted_cluster]

print("Predicted Cluster:", predicted_cluster)
print("Membership Probability:", round(membership_probability, 4))

Predicted Cluster: 1
Membership Probability: 0.9998


In [12]:
# Create final prediction result

prediction_result = {
    "Cluster": int(predicted_cluster),
    "Membership_Probability": round(float(membership_probability), 4)
}

prediction_result

{'Cluster': 1, 'Membership_Probability': 0.9998}

In [13]:
# Display membership probability for each cluster

for i, probability in enumerate(probabilities[0]):
    print(f"Cluster {i}: {probability:.4f}")

Cluster 0: 0.0002
Cluster 1: 0.9998


In [14]:
# Create multiple new customers

new_customers = pd.DataFrame({
    "Recency": [25, 150, 60],
    "Frequency": [8, 1, 5],
    "Monetary": [4200, 350, 2000]
})

new_customers

,Recency,Frequency,Monetary
0,25,8,4200
1,150,1,350
2,60,5,2000


In [15]:
# Apply log transformation

new_customers_log = np.log1p(new_customers)

new_customers_log

,Recency,Frequency,Monetary
0,3.258097,2.197225,8.343078
1,5.017280,0.693147,5.860786
2,4.110874,1.791759,7.601402


In [16]:
# Apply the saved scaler

new_customers_scaled = scaler.transform(new_customers_log)

new_customers_scaled

array([[-0.42730728,  1.24686844,  1.39436188],
       [ 0.88541159, -0.95521426, -0.57838368],
       [ 0.20904306,  0.65323695,  0.8049319 ]])

In [17]:
# Predict clusters for all new customers

predicted_clusters = model.predict(new_customers_scaled)

predicted_clusters

c:\Users\Greesha Vaishnavi\.conda\envs\clust\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but GaussianMixture was fitted with feature names
  warnings.warn(


array([1, 0, 1])

In [18]:
# Get membership probabilities

customer_probabilities = model.predict_proba(new_customers_scaled)

customer_probabilities

c:\Users\Greesha Vaishnavi\.conda\envs\clust\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but GaussianMixture was fitted with feature names
  warnings.warn(


array([[2.36083164e-04, 9.99763917e-01],
       [9.98764759e-01, 1.23524143e-03],
       [9.29497902e-02, 9.07050210e-01]])

In [19]:
# Create prediction results

prediction_results = new_customers.copy()

prediction_results["Predicted_Cluster"] = predicted_clusters

prediction_results["Membership_Probability"] = [
    probabilities.max() for probabilities in customer_probabilities
]

prediction_results

,Recency,Frequency,Monetary,Predicted_Cluster,Membership_Probability
0,25,8,4200,1,0.999764
1,150,1,350,0,0.998765
2,60,5,2000,1,0.907050


In [20]:
# Create reusable prediction function

def predict_customer(recency, frequency, monetary):

    # Create customer RFM data
    customer = pd.DataFrame({
        "Recency": [recency],
        "Frequency": [frequency],
        "Monetary": [monetary]
    })

    # Apply log transformation
    customer_log = np.log1p(customer)

    # Apply saved scaler
    customer_scaled = scaler.transform(customer_log)

    # Predict cluster
    predicted_cluster = model.predict(customer_scaled)[0]

    # Get membership probabilities
    probabilities = model.predict_proba(customer_scaled)[0]

    # Get probability of predicted cluster
    membership_probability = probabilities[predicted_cluster]

    return {
        "Cluster": int(predicted_cluster),
        "Membership_Probability": float(membership_probability)
    }

In [21]:
# Test prediction function

result = predict_customer(
    recency=25,
    frequency=8,
    monetary=4200
)

result

c:\Users\Greesha Vaishnavi\.conda\envs\clust\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but GaussianMixture was fitted with feature names
  warnings.warn(
c:\Users\Greesha Vaishnavi\.conda\envs\clust\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but GaussianMixture was fitted with feature names
  warnings.warn(


{'Cluster': 1, 'Membership_Probability': 0.9997639168360254}

In [22]:
# Test another customer

result = predict_customer(
    recency=150,
    frequency=1,
    monetary=350
)

result

c:\Users\Greesha Vaishnavi\.conda\envs\clust\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but GaussianMixture was fitted with feature names
  warnings.warn(
c:\Users\Greesha Vaishnavi\.conda\envs\clust\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but GaussianMixture was fitted with feature names
  warnings.warn(


{'Cluster': 0, 'Membership_Probability': 0.9987647585733024}

In [23]:
from E_Commerce_Customer_Segmentation.pipeline.stage_06_prediction import PredictionPipeline

In [26]:
import os

# Move from research folder to project root
os.chdir("..")

print("Current working directory:", os.getcwd())

Current working directory: c:\Users\Greesha Vaishnavi\Desktop\dsprojects\E_Commerce_Customer_Segmentation


In [27]:
prediction_pipeline = PredictionPipeline()

In [28]:
result = prediction_pipeline.main(
    recency=25,
    frequency=8,
    monetary=4200
)

result

[2026-10-02 00:57:54,518: INFO: stage_06_prediction: >>>>>> Prediction Stage Started <<<<<<]
[2026-10-02 00:57:54,529: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-02 00:57:54,537: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-02 00:57:54,538: INFO: common: created directory at: artifacts]
[2026-10-02 00:57:54,541: INFO: common: created directory at: artifacts/prediction]
[2026-10-02 00:57:54,576: INFO: stage_06_prediction: >>>>>> Prediction Stage Completed Successfully <<<<<<]


c:\Users\Greesha Vaishnavi\.conda\envs\clust\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but GaussianMixture was fitted with feature names
  warnings.warn(
c:\Users\Greesha Vaishnavi\.conda\envs\clust\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but GaussianMixture was fitted with feature names
  warnings.warn(


{'Cluster': 1, 'Membership_Probability': 0.9997639168360254}